# 🖼️ 02 — Image Preprocessing Pipeline

**EvaliSense — AI-Powered Handwritten Examination Evaluation**

This notebook demonstrates the image preprocessing pipeline that prepares handwritten answer sheets for HTR:

1. **Grayscale conversion** — Reduce colour channels
2. **Noise reduction** — Gaussian and bilateral filtering
3. **Binarization** — Adaptive thresholding (Otsu / adaptive Gaussian)
4. **Deskewing** — Correct rotated scans
5. **Border removal** — Clean margins

The preprocessing module was the first component built in the existing codebase.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from preprocessing import ImagePreprocessor

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

preprocessor = ImagePreprocessor()
print('ImagePreprocessor initialized')
print(f'Module location: {ImagePreprocessor.__module__}')

## 1. Process a Test Image

Apply the full preprocessing pipeline to one of our generated test images.

In [ ]:
# Load a test image
test_image_path = Path('../data/raw/answer_photosynthesis_good.png')

if test_image_path.exists():
    original = cv2.imread(str(test_image_path))
    print(f'Original image shape: {original.shape}')
    print(f'Original image size: {original.nbytes / 1024 / 1024:.1f} MB')
    
    # Run preprocessing
    output_path = Path('../data/processed/test_preprocessed.png')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result = preprocessor.process(str(test_image_path), str(output_path))
    
    print(f'\nProcessed image shape: {result.final.shape}')
    print(f'Metadata: {result.metadata}')
else:
    print(f'Test image not found at {test_image_path}')
    print('Run: python create_test_images.py')

In [ ]:
# Visualize before/after
if test_image_path.exists():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))
    
    # Original
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    ax1.imshow(original_rgb[:1500, :, :])
    ax1.set_title(f'Original ({original.shape[1]}×{original.shape[0]})', fontsize=14)
    ax1.axis('off')
    
    # Preprocessed
    if result.final.ndim == 2:
        ax2.imshow(result.final[:1500, :], cmap='gray')
    else:
        ax2.imshow(cv2.cvtColor(result.final[:1500, :], cv2.COLOR_BGR2RGB))
    ax2.set_title(f'Preprocessed ({result.final.shape[1]}×{result.final.shape[0]})', fontsize=14)
    ax2.axis('off')
    
    plt.suptitle('Image Preprocessing: Before vs After', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/results/preprocessing_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Preprocessing Steps Breakdown

Let's visualize each individual preprocessing step.

In [ ]:
if test_image_path.exists():
    img = cv2.imread(str(test_image_path))
    crop_h = 1200  # Show top portion for clarity
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Step 1: Original
    axes[0, 0].imshow(cv2.cvtColor(img[:crop_h], cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title('1. Original Image')
    axes[0, 0].axis('off')
    
    # Step 2: Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    axes[0, 1].imshow(gray[:crop_h], cmap='gray')
    axes[0, 1].set_title('2. Grayscale Conversion')
    axes[0, 1].axis('off')
    
    # Step 3: Gaussian blur (noise reduction)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    axes[0, 2].imshow(blurred[:crop_h], cmap='gray')
    axes[0, 2].set_title('3. Gaussian Blur (Noise Reduction)')
    axes[0, 2].axis('off')
    
    # Step 4: Otsu thresholding
    _, otsu = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    axes[1, 0].imshow(otsu[:crop_h], cmap='gray')
    axes[1, 0].set_title('4. Otsu Binarization')
    axes[1, 0].axis('off')
    
    # Step 5: Adaptive thresholding
    adaptive = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 15, 10)
    axes[1, 1].imshow(adaptive[:crop_h], cmap='gray')
    axes[1, 1].set_title('5. Adaptive Gaussian Threshold')
    axes[1, 1].axis('off')
    
    # Step 6: Morphological cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    cleaned = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel)
    axes[1, 2].imshow(cleaned[:crop_h], cmap='gray')
    axes[1, 2].set_title('6. Morphological Cleaning')
    axes[1, 2].axis('off')
    
    plt.suptitle('Preprocessing Pipeline — Step-by-Step', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/results/preprocessing_steps.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Process IAM Images

Let's apply preprocessing to real handwriting from the IAM dataset.

In [ ]:
iam_dir = Path('../data/external/iam_lines/test')
iam_images = sorted(iam_dir.glob('*.png'))[:6] if iam_dir.exists() else []

if iam_images:
    fig, axes = plt.subplots(3, 2, figsize=(16, 10))
    fig.suptitle('IAM Images — Original vs Preprocessed', fontsize=16, fontweight='bold')
    
    for idx in range(3):
        img = cv2.imread(str(iam_images[idx]))
        
        # Original
        axes[idx, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx, 0].set_title(f'Original: {iam_images[idx].name}', fontsize=10)
        axes[idx, 0].axis('off')
        
        # Preprocessed
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        axes[idx, 1].imshow(binary, cmap='gray')
        axes[idx, 1].set_title('Preprocessed (Otsu binarization)', fontsize=10)
        axes[idx, 1].axis('off')
    
    plt.tight_layout()
    plt.savefig('../data/results/iam_preprocessing.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No IAM images found in test split.')

## Summary

The preprocessing pipeline handles:
- Grayscale conversion and noise reduction
- Adaptive binarization for varying lighting conditions
- Morphological operations for stroke cleanup
- Deskewing for rotated scans

**Next:** Proceed to `03_htr_recognition.ipynb` for handwritten text recognition.